# Results for model: glm-5

In [ ]:
import polars as pl
import xgboost as xgb
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

DATA_PATH = './jane-street-real-time-market-data-forecasting/train.parquet'
TARGET = 'responder_6'
TOP_QUANTILE = 0.85
ROLLING_WINDOW = 10


def main():
    print("Loading data with Polars...")
    df = pl.read_parquet(DATA_PATH)
    print(f"Data shape: {df.shape}")
    print(f"Columns: {len(df.columns)}")
    
    print("\nPerforming feature engineering...")
    unique_dates = df.select('date_id').unique().sort('date_id').to_series().to_list()
    print(f"Unique dates: {len(unique_dates)}")
    
    date_quantiles = df.group_by('date_id').agg(
        pl.col('feature_00').quantile(TOP_QUANTILE).alias('date_quantile')
    ).sort('date_id')
    
    date_quantiles = date_quantiles.with_columns(
        pl.col('date_quantile').rolling_mean(window_size=ROLLING_WINDOW).alias('rolling_quantile')
    )
    date_quantiles = date_quantiles.fill_null(strategy='forward')
    
    df = df.join(date_quantiles, on='date_id', how='left')
    
    df = df.with_columns([
        (pl.col('feature_00') >= pl.col('rolling_quantile')).cast(pl.Int8).alias('feat_above_top_quantile'),
        ((pl.col('feature_00') - pl.col('rolling_quantile')) / (pl.col('rolling_quantile').abs() + 1e-8)).alias('feat_rel_to_quantile'),
        (pl.col('feature_00') * pl.col('rolling_quantile')).alias('feat_quantile_interaction')
    ])
    
    base_features = [c for c in df.columns if c.startswith('feature_') and c != 'feature_00_quantile' and c != 'date_quantile' and c != 'rolling_quantile']
    engineered_features = ['feat_above_top_quantile', 'feat_rel_to_quantile', 'feat_quantile_interaction']
    all_features = base_features + engineered_features
    
    print(f"Total features: {len(all_features)}")
    
    split_idx = int(len(unique_dates) * 0.8)
    train_dates = set(unique_dates[:split_idx])
    val_dates = set(unique_dates[split_idx:])
    
    train_df = df.filter(pl.col('date_id').is_in(train_dates))
    val_df = df.filter(pl.col('date_id').is_in(val_dates))
    
    print(f"Train samples: {train_df.shape[0]}")
    print(f"Validation samples: {val_df.shape[0]}")
    
    X_train = train_df.select(all_features).to_numpy()
    y_train = train_df.select(TARGET).to_numpy().ravel()
    
    X_val = val_df.select(all_features).to_numpy()
    y_val = val_df.select(TARGET).to_numpy().ravel()
    
    X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
    X_val = np.nan_to_num(X_val, nan=0.0, posinf=0.0, neginf=0.0)
    
    print(f"y_train NaN count: {np.isnan(y_train).sum()}")
    y_train = np.nan_to_num(y_train, nan=0.0)
    y_val = np.nan_to_num(y_val, nan=0.0)
    
    print("\nTraining XGBoost Regressor...")
    model = xgb.XGBRegressor(
        n_estimators=1000,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        colsample_bylevel=0.8,
        min_child_weight=1,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        tree_method='hist',
        enable_categorical=False,
        eval_metric='rmse',
        early_stopping_rounds=50
    )
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=50
    )
    
    print("\nEvaluating model...")
    train_preds = model.predict(X_train)
    val_preds = model.predict(X_val)
    
    train_rmse = np.sqrt(mean_squared_error(y_train, train_preds))
    val_rmse = np.sqrt(mean_squared_error(y_val, val_preds))
    train_r2 = r2_score(y_train, train_preds)
    val_r2 = r2_score(y_val, val_preds)
    
    print(f"Training RMSE: {train_rmse:.6f}")
    print(f"Validation RMSE: {val_rmse:.6f}")
    print(f"Training R2: {train_r2:.6f}")
    print(f"Validation R2: {val_r2:.6f}")
    
    print("\nTop 15 Feature Importances:")
    importance_scores = model.feature_importances_
    feature_importance = list(zip(all_features, importance_scores))
    feature_importance.sort(key=lambda x: x[1], reverse=True)
    
    for feat, imp in feature_importance[:15]:
        print(f"  {feat}: {imp:.6f}")
    
    model.save_model('js_market_forecast_model.json')
    print("\nModel saved to 'js_market_forecast_model.json'")
    
    print("\nGenerating validation predictions...")
    val_results = val_df.select(['date_id', 'time_id', 'symbol_id']).with_columns(
        pl.Series(name='responder_6_pred', values=val_preds),
        pl.Series(name='responder_6_actual', values=y_val)
    )
    
    val_results.write_parquet('validation_predictions.parquet')
    print("Validation predictions saved to 'validation_predictions.parquet'")


if __name__ == '__main__':
    main()